# Exploratory Data Analysis in R

In this workshop we will cover Exploratory Data Analysis (EDA) in R.

## Learning Outcomes

By the end of this workshop, you will be able to

- Load data into a dataframe
- Assess general characteristics of your data (e.g. data types, column names)
- Generate descriptive statistics
- Quantify missingness and outliers
- Visualize univariate, bivariate and multivariate data.
- Identify potential relationships of interest which could be explored using modeling



We will use a dataset which has been handed to us by Barbara who works at NC State University. She has asked us to determine the following:

1. What columns are in the data and what are their values?
2. Are there any problems with the data? Outliers? Missing Values? Data type issues?
3. What variance to you see in the individual values?
4. What is the covariance between variables in the dataset?
5. Does covariance between any variables lead to an interesting model?

Barbara would like reporting and visualization of the data. We have no additional information about the data except a post-it note that was left on our desk.

![post-it note](https://github.com/NCSU-Libraries/intro-to-prog-r/blob/main/assets/notebooks/post_it_note_small.png?raw=true)

### Load Packages


In [ ]:
# If tidyverse isn't installed, install it
if (!require(tidyverse)) {
    install.packages("tidyverse")
}

# load tidyverse
library(tidyverse)

### Load Data

In [ ]:
# Create variable to hold url to raw data in github
mascot.url <- "https://github.com/NCSU-Libraries/intro-to-prog-r/raw/refs/heads/main/RStudio%20Materials/data/NCSU_Mascots_v1.csv"

# read in data
mascot.data <- read.csv(mascot.url)

# Look at first few rows of data
head(mascot.data)

In [ ]:
# look at the last few rows
tail(mascot.data, 4) # we can set the number of rows shown

### Summarize Data Shape & Type

We can use [`glimpse()`](https://www.rdocumentation.org/packages/pillar/versions/1.6.2/topics/glimpse) to get the shape of our data (# of rows and columns), column names, data types and first few values for each column.

In [ ]:
glimpse(mascot.data)

Initial observations of the data include:

- We have 50 observations (rows)
- We have 22 variables (cols) for each observation
- Some variables seem to be missing values.

We initially categorize our variables into two broad data types:

<table>
<tr>
<th> Numeric </th>
<th> Character </th>
</tr>
<tr>
<td>

- Hire.Year
- Termination.Year
- Age
- Height
- Offer.Sent
- Starting.Salary
- Biter
- Hired


</td>
<td>

- Name
- Species
- Breed
- Prior.Position
- Prior.Employer
- Interview.Date
- Hire.Date
- Termination.Date
- Termination.Reason
- Post.Employment.Position
- Nick.Name
- Temperment
- Fur.Color
- Weight

</td>
</tr>
</table>

Next steps:
- identify variables for which we think missing data might be problematic
- assess whether the data type associated with a column make sense for the data stored in it
- explore values in data

## Summarize Data Values

To get information about the values in our data, we can use [`summary()`](https://www.rdocumentation.org/packages/base/versions/3.6.2/topics/summary)

In [ ]:
summary(mascot.data)

#### A NOTE ABOUT BOOLEAN DATA (0/1)

Fields that contain boolean values (0/1) show interesting characteristics in a summary:
- `Min.` will be 0 and `Max.` will be 1
- The mean will represent the percentage of rows for which the value is set to 1

For instance, The mean for `Hired` is 0.24 which means 24% of the rows have `Hired` set to 1.



What observations can we make based on the summary of the values?

Observations:



We may want to get a sense of how many NAs or blanks are in our data.

In [ ]:
# Quantify the number of NAs and blanks in the data
missing_counts_dplyr <- mascot.data %>%
  summarise(across(everything(), ~sum(is.na(.) | . == "")))

print(missing_counts_dplyr)

Before investigating the values of character data, let's convert `Weight` to a numeric value, the date fields from text to dates, and `Offer.Sent`, `Biter`, and `Hired` to factors and provide them with appropriate labels.

We will keep both the old field and the new field, rather than replacing the old field.

### Convert weight field from text to numeric
We'll save the existing Weight column with a new name and then convert the data in the Weight column to numeric.

In [ ]:
# Convert Weight
  # Save Weight column with a new name
mascot.data$Weight.text <- mascot.data$Weight

  # Convert Weight Column to numeric value
mascot.data$Weight <- as.numeric(mascot.data$Weight)

  # Check converted data
glimpse(mascot.data$Weight)
summary(mascot.data$Weight)

### Convert date fields from text to dates
We'll first save text data in a separate field and then use [`as.Date()`](https://www.rdocumentation.org/packages/base/versions/3.3.0/topics/as.Date) to a proper date.

In [ ]:
# Interview.Date

  # Save date column with new name
mascot.data$Interview.Date.text <- mascot.data$Interview.Date

  # Convert Interview.Date to date format
mascot.data$Interview.Date <- as.Date(mascot.data$Interview.Date,
                                      format = "%Y-%m-%d")

  # Check converted data
glimpse(mascot.data$Interview.Date)
summary(mascot.data$Interview.Date)

**TRY IT!**

Using the code above as an example, save the text fields for `Hire.Date` and `Termination.Date` and convert both to a date.

In [ ]:
# Hire.Date

  # Save date column with new name

  # Convert Hire.Date to date format

  # Check converted data


In [ ]:
# Termination.Date

  # Copy date column with new name

  # Convert Termination.Date to date format
                                      format = "%Y-%m-%d")

  # Check converted data


### Convert fields with boolean flags to factors
We will convert `Offer.Sent`, `Biter` and `Hired` to R factors using [`factor()`](https://www.rdocumentation.org/packages/base/versions/3.6.2/topics/factor). Factors let you associate a level and a label to a categorical variable. Converting a categorical variable to a factor can simplify analysis and visualization in R.

In [ ]:
# Convert Offer.Sent, Biter, and Hired to Factors
# We only need to create a new column

# Offer.Sent
mascot.data$Offer.Sent.Factor <- factor(mascot.data$Offer.Sent,
                                        levels = c(0,1),
                                        labels = c("Offer Not Sent",
                                                   "Offer Sent"))
  # Check converted data
glimpse(mascot.data$Offer.Sent.Factor)
summary(mascot.data$Offer.Sent.Factor)
sum(mascot.data$Offer.Sent)

**TRY IT!**

Using the code above as an example, convert `Biter` and `Hired` to factors. Provide appropriate level labels for both variables.

In [ ]:
# Biter

  # Check converted data


In [ ]:

  # Hired

  # Check converted data


We can do an initial assessment of character data. Do do this we need to identify a subset of columns that store character information.

In [ ]:
# identify character fields in mascot.data
char.fields <- mascot.data %>%
  select(where(is.character)) # a smaller df with character columns
print(names(char.fields))


We need to do one more step which is to remove the columns ending with ".text" which we created.

In [ ]:
# Remove fields whose name ends with ".text"
char.fields <- char.fields %>%
  select(!(ends_with(".text")))

# Extract only the column names
char.fields <- names(char.fields) # a vector/list of column names
print(char.fields)

A first step in exploring character data is to quantify how many unique values and blanks (e.g. missing values) each character field has.

If a column has a lot of blanks, we might want to assess whether we need to get that data or not. The number of unique values will give us an idea of what character columns represent potential groups. Also, columns with a lot of unique values will be harder to visualize.

The following code will create a table of character variables, the number of unique values and the number of blanks each variable has and then sorts the table.

In [ ]:
# initialize some vectors to hold our counts
num.blanks <- c()
num.uniq <- c()

# Find the number of blanks and unique values in each field
for (col_name in char.fields) {

  # calculate # of blanks and append to num.blanks vector
  num.blanks <- append(num.blanks,sum(mascot.data[[col_name]]==""))

  # calculate # of unique values and append to num.uniq vector
  num.uniq <- append(num.uniq,
                     length(unique(mascot.data[[col_name]])))
}

# create dataframe that contains 3 columns: column name,
# num.uniq and num.blanks
count.df <- data.frame(Columns = char.fields,
                       uniq.cnt = num.uniq,
                       blanks.cnt = num.blanks)

# sort data by uniq count then blanks.cnt
count.df <- count.df %>%
  arrange(uniq.cnt, blanks.cnt)
print(count.df)

What observations can we make about the character fields in our data?


Possible observations:



Because they have a small number of unique values, we can quickly get a sense of what values are associated with `Post.Employment.Position`, `Termination.Reason`, `Fur.Color` using the [`table()`](https://www.rdocumentation.org/packages/base/versions/3.6.2/topics/table) function. Example code using table with `Post.Employment.Position` is shown below.

In [ ]:
print(toupper("Values in Post.Employment.Position"))
table(mascot.data$Post.Employment.Position)

**TRY IT!**

Using the code above as a template, identify the unique values for `Termination.Reason` and `Fur.Color`.

In [ ]:
print(toupper("Values in Termination.Reason"))


In [ ]:
print(toupper("Values in Fur.Color"))


`Nick.Name` and `Temperment` have a moderate number of unique values, but we can still use table to get a sense of the values in each column.

In [ ]:
print(toupper("Values in Nick.Name"))
table(mascot.data$Nick.Name)

In [ ]:
print(toupper("Values in Temperment"))
table(mascot.data$Temperment)

`Prior.Employer` and `Prior.Position` both have almost as many unique values as rows. We can just list the unique values for each.

We can use [`dplr`](https://r4ds.hadley.nz/data-transform.html) to extract the rows with distinct values, and select only the `Prior.Employer` column.

In [ ]:
# List unique values in Prior Employer
mascot.data %>%
  distinct(Prior.Employer, .keep_all = TRUE) %>%
  select(Prior.Employer)

**TRY IT!**

Using the code above as a template, list the unique values in `Prior.Position`.

In [ ]:
# List unique values of Prior Position


## Visualize Data
### Univariate Plots

#### Continuous
When looking at univariate continuous data, our ultimate goal is to explore the variation in our data. We can do this using histograms or boxplots.

The fields in our data that are continuous are:

- Age
- Height
- Weight
- Starting.Salary

Histograms provide the shape of the data distribution. Boxplots provide a visualization of key summary statistics. A review of the elements included in a boxplot are shown below.

![boxplot summary](https://github.com/NCSU-Libraries/intro-to-prog-r/blob/main/assets/notebooks/boxplot_features_2.png?raw=true)

In [ ]:
# make a histogram of Weight
hist(mascot.data$Weight,
    #  breaks = 50,
     xlab = "Weight",
     ylab = "Frequency")


What observations can we make from the histogram of `Weight`?

Possible observations:



Next steps:
We should look at a histogram for values < 200 and identify which rows are associated with the outliers.

In [ ]:
# Create histogram of Weight < 200
weight.subset <- mascot.data %>%
  filter(Weight < 200)

hist(weight.subset$Weight,
     breaks = seq(0, 150, by = 10),
     prob = TRUE)

In [ ]:
# Identify Weight outliers show Name, Species, Breed and Weight
mascot.data[mascot.data$Weight > 200, c('Name', 'Species', 'Breed', 'Weight')]

What observations can we make about the distribution of weights < 200 of these outliers?

Possible observations:



We can use a boxplot to explore the distribution in height.

In [ ]:
# Generate a boxplot of Height
boxplot(mascot.data$Height,
        xlab = 'Height [in]')

What observations can we make?

Possible observations:



** TRY IT!**

Using code provided above, generate a list of the height outliers visible in the boxplot.


In [ ]:
# List of height outliers Show Name, Species, Breed and Height
mascot.data[mascot.data$Height > 40, c('Name', 'Species', 'Breed', 'Height')]

**TRY IT!**

Using code above, create a histogram and boxplot of `Starting.Salary` and `Age`. If there are outliers, extract them into a table.

In [ ]:
# Histogram of Salary


In [ ]:
# Boxplot of starting salary


In [ ]:
# histogram of age


In [ ]:
# Boxplot of Age


In [ ]:
# Extract Age outlier


What observations can we make about the height outliers, and histogram and boxplot of starting salary and age? Age Outliers? 

Possible observations:
1. The height outliers are probably not an error. We expect horses to be tall. It is likely that the value for Banjo, the St. Bernard is also probably appropriate.
2. No starting salaries in the range 9900-10000 or 10200-10300
3. The median starting salary is about 9900 and there is a slight left skew
4. 50% of the starting salaries are between 9600-10100
5. Starting salary shows no outliers.
6. Most mascots in the data are aged 9-11
7. Median age is about 10.5 years.
8. There is one outlier in the lower 1.5*IQR.

#### Categorical

We use barplots for univariate categorical data. The following columns in our data frame are categorical and in order of number of unique values:

<table>
<tr>
<td>

1. Post-Employment.Position
1. Termination.Reason
1. Fur.Color
1. Temperment
1. Nick Name
1. Species
1. Prior.Employer
1. Prior.Position
1. Breed
1. Name
</td>
<td>



</td>
</tr>
</table>

We'll start with columns that don't have a lot of unique values.

We won't bother making plots of
`Nick.Name` or `Name` since they are labels for the data.

We use the [`barplot()`](https://www.rdocumentation.org/packages/graphics/versions/3.6.2/topics/barplot) function to generate a barplot.

In [ ]:
# Create a barplot of Post.Employment.Position
barplot(table(mascot.data$Post.Employment.Position))

We can tweak base R plots by setting graphical parameters using the [`par()`](https://www.rdocumentation.org/packages/graphics/versions/3.6.2/topics/par) function.

In [ ]:
# Create a barplot of Termination.Reason
# save default par() values
opar <- par()

# change borders on plot by modifying
# the mar parameter
par(mar = c(4,7,1,1))
# mar = c(bottom, left, top, right)
# default par(mar = c(5.1, 4.1, 4.1, 2.1))
# unit in lines of text

# Generate barplot
barplot(table(mascot.data$Termination.Reason),
        horiz = TRUE,
        las = 1,
        cex.names = .75)

# reset par()
suppressWarnings(par(opar))

In [ ]:
barplot(table(mascot.data$Fur.Color),
              cex.names = .85)

In [ ]:
# Create barplot of Temperment
barplot(table(mascot.data$Temperment),
              xlab = "Temperment",
              horiz = TRUE,
              las = 1,
              cex.names = .75)

An improvement to the chart of `Temperment` would be to sort from largest to smallest values. We can use the [`dplyr`](https://r4ds.hadley.nz/data-transform.html) package in `tidyverse` to accomplish this.

In [ ]:
# Group data by breed and create counts
val.counts <- mascot.data %>%
  group_by(Temperment) %>%
  count() %>%
  arrange(n)

# Save default par() values
opar <- par()
# Set par()
par(mar = c(4,7,1,1))

barplot(val.counts$n,
              names.arg = val.counts$Temperment,
              xlab="Temperment",
              horiz=TRUE,
              las = 1,
              cex.names = .75)
suppressWarnings(par(opar))

**TRY IT!**

Using the code used to generated the sorted plot for counts of `Temperment`, create a similar visualization for `Species` and `Breed`.

In [ ]:
# grab counts of Species and sort data
# to get largest values on top need ascending sort
# arrange defaults to ascending sort


In [ ]:
# Barplot of Breed

# Group data by breed and create counts


#### Factors

We defined the following factors:
- Offer.Sent.Factor
- Biter.Factor
- Hired.Factor

We visualize factors like other categorical columns.

In [ ]:
# Create barplot of Offer.Sent
barplot(table(mascot.data$Offer.Sent.Factor))

**TRY IT!**

Using code above, make a `barplot` of `Biter.Factor` and `Hired.Factor`

In [ ]:
# Create barplot of Biter


In [ ]:
# Create barplot of Hired


#### Dates
Dates can be visualized as categorical variables by grouping them. Grouping over day, month, year or a combination may be informative.

Dates in our data include:
- Interview.Date
- Hire.Date
- Termination.Date

Hire.Date and Termination.Date are grouped in the data in the fields Hire.Year and Termination.Year

In [ ]:
# Create a barplot of Hire.Year
barplot(table(mascot.data$Hire.Year),
        main="Hire Year")

Use the code above to generate a plot of Termination.Year.

In [ ]:
# Create barplot of Termination.Year
barplot(table(mascot.data$Termination.Year),
        main="Termination Year")

To create a barplot of Interview.Date by year, we need to use the `lubridate` function `year()`.

In [ ]:
# Create a barplot of Interview.Date grouped by Year
barplot(table(year(mascot.data$Interview.Date)),
        main="Interview Year")

In [ ]:
# Create a barplot of Interview.Date grouped by Month
barplot(table(month(mascot.data$Interview.Date, label = TRUE)),
        main="Interview Month")

In [ ]:
# Create a barplot of Interview.Date grouped by Month
barplot(table(wday(mascot.data$Interview.Date, label = TRUE)),
        main="Interview Day")

### Bivariate Plots

#### [Pair Plots](https://intro2r.com/simple-base-r-plots.html#pairs-plots) (2 Numeric Columns)

A pair plot generates a scatter plot between each numeric column in the data passed to `pairs()`. The three columns that we converted to factors, `Hired`, `Offer.Sent`, and `Biter` are still in our data in numeric form, so they are included in our pair plot.

In [ ]:
num.cols <- mascot.data %>%
  select(where(is.numeric)) # a smaller df with character columns
names(num.cols)

pairs(num.cols)


In a large dataset, we might focus on a subset of fields that are of interest. Using the code above, create a subset of data using the columns `Hired`, `Starting.Salary`, `Age`, `Height` and `Weight` and create a pairs plot from this subset. Hint: you can pass the names of the columns you want to keep to the `select()` function.

In [ ]:
# Select the subset of columns for the pair plot
num.cols <- mascot.data %>%
  select('Hired', 'Starting.Salary', 'Age', 'Height', 'Weight') # a smaller df with character columns
names(num.cols)

# Generate the pair plot
pairs(num.cols)


#### [Scatter Plots](https://intro2r.com/simple-base-r-plots.html#scatterplot) (2 Numeric Columns)

Once we investigate pair plots, we can use scatter plots to focus on continuous values of interest.

In [ ]:
plot(mascot.data$Age,mascot.data$Starting.Salary)


#### [Barcharts](https://www.datacamp.com/doc/r/bar) (1 Categorical 1 Numeric)

In [ ]:
data <- mascot.data %>%
  group_by(Hired.Factor) %>%
  summarise(Salary = mean(Starting.Salary, na.rm = TRUE))

barplot(data$Salary,
        names.arg = data$Hired.Factor
        )


#### [Grouped Barcharts](https://www.datacamp.com/doc/r/bar) (2 Categorical 1 Numeric)

In [ ]:
counts <- table(mascot.data$Hired.Factor, mascot.data$Temperment)

opar <- par()
par(mar = c(2.5, 7, 2, 2))
barplot(counts,
        main = "Counts of Hired by Temperment",
        xlab="Temperment", col=c("blue","red"),
        legend = rownames(counts),
        horiz = TRUE,
        las = 1,
        cex.lab = .75)
suppressWarnings(par(opar))

### Multivariate Plots

#### [Scatter Plots](https://www.sthda.com/english/wiki/scatter-plots-r-base-graphs)

In [ ]:
# Using a factor
plot(mascot.data$Weight, mascot.data$Height,
     pch = 19,
     col = mascot.data$Offer.Sent.Factor)
legend('bottomright',
       col = c('red','black'),
       pch = 19,
       legend = levels(mascot.data$Offer.Sent.Factor),
       bty = 'n')

In [ ]:
# Using a categorical which is not a factor
plot(mascot.data$Weight, mascot.data$Height,
     pch = 19,
     col = as.factor(mascot.data$Species))
legend('bottomright',
       legend = levels(as.factor(mascot.data$Species)),
       col = as.factor(levels(as.factor(mascot.data$Species))),
       cex = .6,
       bty = 'n',
       pch = 19)

#### Barplots

We can generate barplots for multivariate data by using grouped or stacked bar charts.

In [ ]:
# generate data by grouping it by two categorical variables
# and summarising a continuous variable
data <- mascot.data %>%
  group_by(Hired.Factor, Temperment) %>%
  summarise(Salary = mean(Starting.Salary, na.rm = TRUE)) %>%
  drop_na() %>%
  filter(Temperment != "")
data


In [ ]:
# Generate plot from data
opar <- par()
par(mar = c(2.5, 7, 2, 2))
barplot(data$Salary,
        names.arg = as.factor(data$Temperment),
        legend = unique(data$Hired.Factor),
        col = c("red","blue"),
        horiz = TRUE,
        las = 1)
suppressWarnings(par(opar))

#### [Coplots](https://intro2r.com/simple-base-r-plots.html#coplots)

In [ ]:
coplot.data <- mascot.data %>%
  drop_na(Weight, Height, Starting.Salary) %>%
  select(Weight,Height,Starting.Salary)
nrow(coplot.data)
summary(coplot.data)

In [ ]:
coplot(Weight ~ Height | Starting.Salary, data = coplot.data)


In [ ]:
given.salary <- co.intervals(coplot.data$Starting.Salary,
                              number=3, overlap=.33)
print(given.salary)
coplot(Weight ~ Height | Starting.Salary, data = coplot.data,
       given.values = given.salary)

## Next Steps

We have explored the data quite a bit. Next steps would be to identify one or two relationships to explore further. This may involve subsetting the data further or using simple models like linear (continuous, fairly normal response) or logistic regression (binary/boolean response).

What relationships or trends do you think might be interesting?

Potential relationships:






Based on the data, what job do you think Barbara has at NC State?

**TRY IT ON YOUR OWN!**

Using the code above try exploring another dataset.



In [ ]:
# Another synthetic dataset
celeb.url <- "https://github.com/NCSU-Libraries/intro-to-prog-r/raw/refs/heads/main/RStudio%20Materials/data/NCSU%20Celebrity%20Graduates_v1.csv"


## Links to additional resources

### R

[R for Data Science](https://r4ds.hadley.nz/)

[Introduction to R](https://intro2r.com/)

### Base R Plots
[Intro2R](https://intro2r.com/graphics_base_r.html)

[STHDA](https://www.sthda.com/english/wiki/r-base-graphs)

